In [6]:
import scanpy as sc
import scarches
from scarches.models.scpoli import scPoli
import scib
import os
import anndata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import re

# Prepare reference

In [7]:
adata = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/6_correct_intagretion//output_260420/Atlas_level1-corrected-hvg-preint.h5ad") # before integration so correct normalization etc. counts is rounded_corrected_counts, still has the unknowns

In [8]:
adata

AnnData object with n_obs × n_vars = 490300 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'cell_type_level1_corrected', 'atlas_key'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'counts', 'log1p_scran_samplewise', 'raw_decontXcounts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [9]:
adata.obs["cell_type_level1_corrected"].value_counts()

cell_type_level1_corrected
T cell                   158997
Neutrophil                46766
unknown                   42096
Macrophage                38157
Fibroblast                34461
Endothelial cell          33793
Smooth muscle cell        31266
B cell                    27868
Monocyte                  24508
Dendritic cell            24137
Natural killer cell       16353
Pericyte                   8526
Plasma cell                2306
Erythrocyte/Erythroid       504
Mast cell                   469
Basophil                     93
Name: count, dtype: int64

In [11]:
# transfer new annotation and remove cells that i removed so we have the correct reference

In [5]:
## 这里adata是整合前的数据（保留了正确的标准化等信息）但还有unkonwn;adata_new有校正完所有细胞类型；把这个细胞类型 给cell_type_level1?

In [10]:
adata_new = sc.read_h5ad("/home/lixiangyu/zr/Annotate/ANNOTATE_new/6_correct_intagretion/./output_260420/Atlas_level1-corrected-hvg-integrated-nouncert-allgenes-names.h5ad") # new pipeline with pdcs and no uncertains

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [11]:
adata_new.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                   162309
Neutrophil                53025
Macrophage                39042
Fibroblast                36601
Endothelial cell          35215
Smooth muscle cell        31752
B cell                    29863
Monocyte                  25813
Dendritic cell            24419
Natural killer cell       16797
Pericyte                   8965
Plasma cell                3264
Basophil                   2244
Erythrocyte/Erythroid      1095
Mast cell                   702
Name: count, dtype: int64

In [12]:
adata.obs["cell_type_level1"].value_counts()

cell_type_level1
T cell                   154507
Neutrophil                80608
Macrophage                44872
Fibroblast                40904
Endothelial cell          34805
B cell                    34139
Smooth muscle cell        31753
Natural killer cell       21348
Dendritic cell            17941
Monocyte                  17343
Pericyte                   8974
Mast cell                  2467
Erythrocyte/Erythroid       533
Basophil                    106
Name: count, dtype: int64

In [13]:
#Subset adata to include only the cells in adata_new
adata = adata[adata_new.obs_names].copy()

# Transfer the "cell_type_level1" column from adata_new to adata
adata.obs['cell_type_level1'] = adata_new.obs['cell_type_level1'].reindex(adata.obs_names).values

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [14]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   162309
Neutrophil                53025
Macrophage                39042
Fibroblast                36601
Endothelial cell          35215
Smooth muscle cell        31752
B cell                    29863
Monocyte                  25813
Dendritic cell            24419
Natural killer cell       16797
Pericyte                   8965
Plasma cell                3264
Basophil                   2244
Erythrocyte/Erythroid      1095
Mast cell                   702
Name: count, dtype: int64

In [15]:
del adata.obs["atlas_key"]
del adata.obs["cell_type_level1_corrected"]

In [16]:
adata

AnnData object with n_obs × n_vars = 471106 × 2000
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'size_factors', 'leiden', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue'
    var: 'original_gene_names', 'highly_variable'
    uns: 'log1p', 'neighbors', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'counts', 'log1p_scran_samplewise', 'raw_decontXcounts', 'uncorrected_counts'
    obsp: 'connectivities', 'distances'

In [17]:
adata.write("./output_all_data260420/Atlas_reference_preint-hvg.h5ad")

In [2]:
adata = sc.read_h5ad("./output_all_data260420/Atlas_reference_preint-hvg.h5ad")

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


# Train reference

In [ ]:
early_stopping_kwargs = {
    "early_stopping_metric": "val_prototype_loss",
    "mode": "min",
    "threshold": 0,
    "patience": 20,
    "reduce_lr": True,
    "lr_patience": 13,
    "lr_factor": 0.1,
}
    
scpoli_model = scPoli(
adata=adata,
condition_keys="sample",
cell_type_keys="cell_type_level1", 
embedding_dims=10,
recon_loss='mse',
)

scpoli_model.train(
    n_epochs=50,
    pretraining_epochs=40,
    early_stopping_kwargs=early_stopping_kwargs,
    eta=5,
)


In [ ]:
scpoli_model.save("./output_all_data260420/model_260420/reference_level1", overwrite=True) # new pipeline with pdcs

In [4]:
from scarches.models import scPoli
scpoli_model = scPoli.load("./output_all_data260420/model_260420/reference_level1", adata=adata)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:63: UserWarning: Since v1.0.0, scvi-tools no longer uses a random seed by default. Run `scvi.settings.seed = 0` to reproduce results from previous versions.
  self.seed = seed
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scvi/_settings.py:70: UserWarning: Setting `dl_pin_memory_gpu_training` is deprecated in v1.0 and will be removed in v1.1. Please pass in `pin_memory` to the data loaders instead.
  self.dl_pin_memory_gpu_training = (
 captum (see https://github.com/pytorch/captum).


RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# 1. 取 scPoli 整合后的 latent representation
scpoli_model.model.eval()

latent = scpoli_model.get_latent(
    adata,
    mean=True
)

# 2. 建一个新的 AnnData 保存 latent
adata_latent = sc.AnnData(latent)
adata_latent.obs = adata.obs.copy()

# 3. 基于 latent 计算 UMAP
sc.pp.neighbors(adata_latent, n_neighbors=15, use_rep="X")
sc.tl.umap(adata_latent)

# 4. 看 batch/sample 是否混合
sc.pl.umap(
    adata_latent,
    color=["sample", "cell_type_level1"],
    wspace=0.4,
    frameon=False
)

NameError: name 'scpoli_model' is not defined